# spd1168x.py bench test

Manual test of the SPD1168X driver against real hardware -- confirms `turn_on()`, `set_voltage()`, `set_current_limit()`, `output_on()`/`output_off()`, and the coil field calibration (`set_field()`, `current_for_field()`, `voltage_for_current()`) actually work before trusting them in an automated experiment flow.

**Before running**: connect the SPD1168X's output to the coil (not left open/shorted carelessly). Start with a small field/current for the very first test on a given setup, and keep an eye on the coil temperature if you run it for a while -- `set_field()` sets the current limit to exactly the target current and gives the voltage setpoint some headroom, so the supply runs in constant-current mode.


## Find the resource address

In [1]:
import pyvisa
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('USB0::0xF4EC::0x1410::SPD13DCQ7R0986::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR')


Copy the SPD1168X's resource string from the list above and paste it below.

In [2]:
import sys
sys.path.insert(0, "C:/Users/codin/Documents/college/diamonds/experiment")

from spd1168x import SPD1168X

PSU_RESOURCE = "USB0::0xF4EC::0x1410::SPD13DCQ7R0986::INSTR"  # update to match rm.list_resources() above

psu = SPD1168X(PSU_RESOURCE, debug=True)
print(psu.idn())

SPD1168X: connected
Siglent Technologies,SPD1168X,SPD13DCQ7R0986,2.1.1.9R1,V1.0


## Test set_voltage() / set_current_limit() individually

Output stays OFF for these -- just confirms the setpoint commands are accepted (`debug=True` prints each SCPI command; `write()` raises if the instrument reports an error).

In [3]:
psu.set_voltage(1.0)
print("voltage setpoint readback:", psu.get_voltage_setpoint())

psu.set_current_limit(0.5)
print("current limit setpoint readback:", psu.get_current_limit_setpoint())

VOLT 1.0 => +0, No error
voltage setpoint readback: 1.0
CURR 0.5 => +0, No error
current limit setpoint readback: 0.5


## Test output_on() / output_off()

Confirm the measured output actually follows -- checks `read_voltage()`/`read_current()` before and after toggling.

In [4]:
import time

print("before output_on: V =", psu.read_voltage(), "I =", psu.read_current())

psu.output_on()
time.sleep(0.5)
print("after output_on:  V =", psu.read_voltage(), "I =", psu.read_current())

before output_on: V = 0.0 I = 0.0
OUTP CH1,ON => +0, No error
after output_on:  V = 0.171 I = 0.5


In [5]:
psu.output_off()
time.sleep(0.5)
print("after output_off: V =", psu.read_voltage(), "I =", psu.read_current())

OUTP CH1,OFF => +0, No error
after output_off: V = 0.0 I = 0.0


## Test turn_on() convenience method

Sets voltage + current limit BEFORE enabling output, then enables it. Using one of the calibration points (1.5 A / 0.509 V, expected ~21.3 G) here -- make sure the coil is actually connected before running this cell.

In [6]:
psu.turn_on(0.509, 1.5)
time.sleep(0.5)
print("voltage setpoint:", psu.get_voltage_setpoint())
print("current limit setpoint:", psu.get_current_limit_setpoint())
print("measured: V =", psu.read_voltage(), "I =", psu.read_current())

psu.turn_off()

VOLT 0.509 => +0, No error
CURR 1.5 => +0, No error
OUTP CH1,ON => +0, No error
SPD1168X: output ON at 0.509 V, 1.5 A limit
voltage setpoint: 0.509
current limit setpoint: 1.5
measured: V = 0.506 I = 1.507
OUTP CH1,OFF => +0, No error
SPD1168X: output OFF


## Sanity-check the calibration fit

`FIELD_PER_AMP`/`FIELD_INTERCEPT` and `COIL_RESISTANCE`/`VOLTAGE_INTERCEPT` come from a linear fit to the 4 calibration points in `spd1168x.py`. Reproduce the fit's predictions at those same currents and compare against the measured calibration data -- should match closely.

In [8]:
from spd1168x import (
    FIELD_PER_AMP, FIELD_INTERCEPT, COIL_RESISTANCE, VOLTAGE_INTERCEPT,
    current_for_field, voltage_for_current,
)

print(f"field fit:   field_G = {FIELD_PER_AMP:.4f} * I_A + {FIELD_INTERCEPT:.4f}")
print(f"voltage fit: V = {COIL_RESISTANCE:.4f} * I_A + {VOLTAGE_INTERCEPT:.4f}  (coil resistance)")

calibration = [(0.172, 0.5, 7.0), (0.341, 1.0, 14.3), (0.509, 1.5, 21.3), (0.678, 2.0, 28.4)]
print(f"{'I (A)':>8} {'field meas (G)':>15} {'field pred (G)':>15} {'V meas':>8} {'V pred':>8}")
for v_meas, i, g_meas in calibration:
    g_pred = FIELD_PER_AMP * i + FIELD_INTERCEPT
    v_pred = voltage_for_current(i)
    print(f"{i:8.2f} {g_meas:15.2f} {g_pred:15.2f} {v_meas:8.3f} {v_pred:8.3f}")

field fit:   field_G = 14.2400 * I_A + -0.0500
voltage fit: V = 0.3372 * I_A + 0.0035  (coil resistance)
   I (A)  field meas (G)  field pred (G)   V meas   V pred
    0.50            7.00            7.07    0.172    0.172
    1.00           14.30           14.19    0.341    0.341
    1.50           21.30           21.31    0.509    0.509
    2.00           28.40           28.43    0.678    0.678


## Test set_field()

Requests a field in gauss, converts to a target current via the calibration fit, and turns the supply on in constant-current mode. If you have a gaussmeter/probe on the coil, compare its reading against the requested field below.

In [9]:
TEST_FIELD_G = 10.0  # pick a modest field for the first real test

psu.set_field(TEST_FIELD_G)
time.sleep(0.5)
print("voltage setpoint:", psu.get_voltage_setpoint())
print("current limit setpoint:", psu.get_current_limit_setpoint())
print("measured: V =", psu.read_voltage(), "I =", psu.read_current())
print("expected field from measured current:",
      FIELD_PER_AMP * psu.read_current() + FIELD_INTERCEPT, "G")

VOLT 0.28977808988764026 => +0, No error
CURR 0.7057584269662915 => +0, No error
OUTP CH1,ON => +0, No error
SPD1168X: output ON at 0.28977808988764026 V, 0.7057584269662915 A limit
SPD1168X: field set to 10.0 G (I=0.7058 A, V=0.2898 V)
voltage setpoint: 0.29
current limit setpoint: 0.706
measured: V = 0.24 I = 0.706
expected field from measured current: 10.003440000000008 G


In [10]:
psu.turn_off()

OUTP CH1,OFF => +0, No error
SPD1168X: output OFF


## Shut down

In [11]:
psu.turn_off()
time.sleep(0.2)
print("after turn_off: V =", psu.read_voltage(), "I =", psu.read_current())

psu.close()
print("closed")

OUTP CH1,OFF => +0, No error
SPD1168X: output OFF
after turn_off: V = 0.0 I = 0.0
SYST:LOCAL => -113,Undefined header,SYST:LOCAL
go_to_local() not supported on this unit, ignoring: SPD1168X error after 'SYST:LOCAL': -113,Undefined header,SYST:LOCAL
closed
